<a href="https://colab.research.google.com/github/jashwanth-cse/Jashwanth-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Phase1_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import numpy as np
warnings.filterwarnings('ignore')
print("pandas version:",pd.__version__)
print("sqlite3 version:",sqlite3.version)

pandas version: 2.2.2
sqlite3 version: 2.6.0


In [3]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#Data ingestion
df=pd.read_csv('/content/drive/MyDrive/spotify_tracks.csv')
print(f"Dataset loaded:{df.shape[0]} tracks,{df.shape[1]} columns")


Dataset loaded:62317 tracks,22 columns


In [5]:
print("--------Data understanding---------------")
print(df.columns.tolist())
#No.of rows & columns
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

#Displays 5 random rows from your dataset.
print(df.sample(5))

print("Column names:",df.columns.tolist())

--------Data understanding---------------
['track_id', 'track_name', 'artist_name', 'year', 'popularity', 'artwork_url', 'album_name', 'acousticness', 'danceability', 'duration_ms', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'track_url', 'language']
Rows: 62317
Columns: 22
                     track_id  \
60941  47gAEoPquQ4AZU3iZ8PSBd   
37558  5C52vhlOmER437CzCf0MLT   
61050  3CXQNZ6Ln8zTcdq4IbKrRK   
9086   3e4fXrR4vXhkdUlTFf65Nx   
17600  1c2dawAraN4X3rknlJQemn   

                                              track_name  \
60941  Chapter 147 - The Boat House - The BRAND NEW p...   
37558                                When I Was Your Man   
61050  Chapter 18 - The Boat House - The BRAND NEW pa...   
9086                                   Rakh Le Uska Naam   
17600                                  Jai Mande Madesha   

                        artist_name  year  popularity  \
60941    Keri Beevis, Shakira 

In [6]:
print("--------Data Cleaning & Preprocessing---------------")

# Missing Values Check
print("\nChecking Missing Values...")
print(df.isnull().sum())

total_missing = df.isnull().sum().sum()
print("\nTotal Missing Values:", total_missing)

if total_missing == 0:
    print("No missing values found.")
else:
    print("Missing values detected.")

# Duplicate Records Check
print("\nChecking Duplicate Records...")
duplicates = df.duplicated().sum()
print("Duplicate Records:", duplicates)

# Remove Duplicates
print("\nRemoving Duplicate Records...")
df = df.drop_duplicates()

print("Duplicate Records After Removal:", df.duplicated().sum())

# Data Types Check
print("\nChecking Data Types...")
print(df.dtypes)

# Rename Columns for Better Readability
print("\nRenaming Columns...")

df.rename(columns={
    'track_id':'song_id',
    'track_name':'song_name',
    'artist_name':'artist',
    'duration_ms':'duration_milliseconds'
}, inplace=True)

print("Updated Column Names:")
print(df.columns.tolist())

# Create New Derived Column
print("\nCreating Duration in Minutes Column...")

df['duration_minutes'] = round(
    df['duration_milliseconds'] / 60000, 2
)

print(df[['duration_milliseconds','duration_minutes']].head())

# Final Validation
print("\nFinal Validation")
print("Dataset Shape:", df.shape)
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Records:", df.duplicated().sum())

# Save Cleaned Dataset
df.to_csv(
    '/content/drive/MyDrive/spotify_tracks_cleaned.csv',
    index=False
)

print("\nCleaned dataset saved successfully!")
print("File: spotify_tracks_cleaned.csv")

--------Data Cleaning & Preprocessing---------------

Checking Missing Values...
track_id            0
track_name          0
artist_name         0
year                0
popularity          0
artwork_url         0
album_name          0
acousticness        0
danceability        0
duration_ms         0
energy              0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
speechiness         0
tempo               0
time_signature      0
valence             0
track_url           0
language            0
dtype: int64

Total Missing Values: 0
No missing values found.

Checking Duplicate Records...
Duplicate Records: 78

Removing Duplicate Records...
Duplicate Records After Removal: 0

Checking Data Types...
track_id             object
track_name           object
artist_name          object
year                  int64
popularity            int64
artwork_url          object
album_name           object
acousticness        float64
dance

In [7]:
print("--------SQLite Database Integration---------------")

# Create SQLite Connection
conn = sqlite3.connect('spotify_analysis.db')

print("Database created successfully!")

# Load DataFrame into SQLite Table
df.to_sql(
    'spotify_tracks',
    conn,
    if_exists='replace',
    index=False
)

print("Dataset loaded into SQLite table successfully!")

# Verify Record Count
query = """
SELECT COUNT(*) AS total_records
FROM spotify_tracks
"""

record_count = pd.read_sql(query, conn)

print("\nVerifying Data Insertion...")
print(record_count)

# Display First 5 Records
query = """
SELECT *
FROM spotify_tracks
LIMIT 5
"""

sample_data = pd.read_sql(query, conn)

print("\nFirst 5 Records from Database:")
print(sample_data)

print("\nDatabase Integration Completed Successfully!")

--------SQLite Database Integration---------------
Database created successfully!
Dataset loaded into SQLite table successfully!

Verifying Data Insertion...
   total_records
0          62239

First 5 Records from Database:
                  song_id                                   song_name  \
0  2r0ROhr7pRN4MXDMT1fEmd                  Leo Das Entry (From "Leo")   
1  4I38e6Dg52a2o2a8i5Q5PW                                AAO KILLELLE   
2  59NoiRhnom3lTeRFaBzOev      Mayakiriye Sirikiriye - Orchestral EDM   
3  5uUqRQd385pvLxC8JX3tXn    Scene Ah Scene Ah - Experimental EDM Mix   
4  1KaBRg2xgNeCljmyxBH1mo  Gundellonaa X I Am A Disco Dancer - Mashup   

                                              artist  year  popularity  \
0                                Anirudh Ravichander  2024          59   
1  Anirudh Ravichander, Pravin Mani, Vaishali Sri...  2024          47   
2           Anirudh Ravichander, Anivee, Alvin Bruno  2024          35   
3  Anirudh Ravichander, Bharath Sankar, K

In [8]:
print("--------SQL Query 1 : Filtering---------------")

query = """
SELECT song_name, artist, language, popularity
FROM spotify_tracks
WHERE popularity > 90
ORDER BY popularity DESC
LIMIT 20;
"""

result = pd.read_sql(query, conn)

print(result)

--------SQL Query 1 : Filtering---------------
         song_name                 artist language  popularity
0        Big Dawgs     Hanumankind, Kalmi  English          93
1  Blinding Lights             The Weeknd  English          91
2          Starboy  The Weeknd, Daft Punk  English          91


In [9]:
print("--------SQL Query 2 : Sorting---------------")

query = """
SELECT song_name, artist, popularity
FROM spotify_tracks
ORDER BY popularity DESC
LIMIT 10;
"""

result = pd.read_sql(query, conn)

print(result)

--------SQL Query 2 : Sorting---------------
                                           song_name  \
0                                          Big Dawgs   
1                                    Blinding Lights   
2                                            Starboy   
3  Bye Bye Bye - From Deadpool and Wolverine Soun...   
4                                          Anti-Hero   
5                                           cardigan   
6                           Something Just Like This   
7                                     Counting Stars   
8  Is It Over Now? (Taylor's Version) (From The V...   
9                                             august   

                       artist  popularity  
0          Hanumankind, Kalmi          93  
1                  The Weeknd          91  
2       The Weeknd, Daft Punk          91  
3                      *NSYNC          90  
4                Taylor Swift          89  
5                Taylor Swift          89  
6  The Chainsmokers, Coldplay 

In [10]:
print("--------SQL Query 3 : Aggregation---------------")

query = """
SELECT
AVG(popularity) AS average_popularity,
MAX(popularity) AS highest_popularity,
MIN(popularity) AS lowest_popularity
FROM spotify_tracks;
"""

result = pd.read_sql(query, conn)

print(result)

--------SQL Query 3 : Aggregation---------------
   average_popularity  highest_popularity  lowest_popularity
0           15.357589                  93                  0


In [11]:
print("--------SQL Query 4 : Group By---------------")

query = """
SELECT
language,
COUNT(*) AS total_songs,
ROUND(AVG(popularity),2) AS avg_popularity
FROM spotify_tracks
GROUP BY language
ORDER BY avg_popularity DESC;
"""

result = pd.read_sql(query, conn)

print(result)

--------SQL Query 4 : Group By---------------
    language  total_songs  avg_popularity
0     Korean         6893           27.55
1      Hindi         5740           18.20
2    English        23389           14.76
3     Telugu          321           14.32
4    Unknown        13005           12.26
5      Tamil        12609           11.91
6  Malayalam          282            7.32


In [12]:
print("--------SQL Query 5 : Conditional Query---------------")

query = """
SELECT
song_name,
artist,
popularity,

CASE
    WHEN popularity >= 80 THEN 'Hit Song'
    WHEN popularity >= 50 THEN 'Average Song'
    ELSE 'Low Popularity'
END AS popularity_category

FROM spotify_tracks
LIMIT 20;
"""

result = pd.read_sql(query, conn)

print(result)

--------SQL Query 5 : Conditional Query---------------
                                     song_name  \
0                   Leo Das Entry (From "Leo")   
1                                 AAO KILLELLE   
2       Mayakiriye Sirikiriye - Orchestral EDM   
3     Scene Ah Scene Ah - Experimental EDM Mix   
4   Gundellonaa X I Am A Disco Dancer - Mashup   
5      Villain Yevadu Ra (From "Leo (Telugu)")   
6                      Gundellonaa - Pop Kuthu   
7                     Knockout Song - Funk Mix   
8                   Rathamaarey (LoFi Version)   
9                     Theeraamal - Ambient Mix   
10                      Nagarathey - Dance Mix   
11       Villain Kaun Hai (From "Leo (Hindi)")   
12      Villain Yaarava (From "Leo (Kannada)")   
13     Villain Aarada (From "Leo (Malayalam)")   
14                  Nagarathey - Synthwave Mix   
15                Knockout Song - Ambient Lofi   
16                   Theeraamal - Ambient Lofi   
17          Ezhundhu Vaa (From "Singappenney"